# 38장 · 재정렬과 다중 질의 검색

재정렬은 기존 후보들의 순서를 바꾸고, 다중 질의 검색은 질문 표현을 늘려 후보를 모읍니다. 첫 검색에 정답 문서가 없는 경우 두 방법의 차이가 드러납니다.

대출 기간을 물었는데 후보 목록 맨 앞에 마침표만 남은 청크가 있습니다. 답을 담은 청크가 후보 안에는 있는지 먼저 확인하세요. 후보 안에서 순서를 고치는 일과 질문 표현을 늘려 다시 찾는 일은 서로 다른 문제를 풉니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## sorted()를 이용한 후보 재정렬

37장의 16글자 청크 중 library-1:0에는 14일이 있고 library-1:1에는 마침표만 있습니다. 순위 차이를 눈으로 보기 위해 이 두 입력의 순서를 뒤집어 둡니다. 여기서 쓰는 점수는 수업용 단어 일치이며 학습된 재순위 모델의 결과와 구분합니다.

### 같은 후보 두 건에서 1위가 바뀌는 과정

**첫 후보에는 마침표만 있습니다.**　대출 기간을 담은 청크는 이미 둘째 자리에 있습니다.

```
1. library-1:1  .
2. library-1:0  도서 대출 기간은 14일입니다
```

**본문에 ‘대출’이 있는지 봅니다.**　첫 청크의 비교값은 False, 둘째 청크는 True입니다. 노트북의 정렬 규칙은 True를 앞으로 보냅니다.

```
library-1:1  False
library-1:0  True
```

**같은 두 후보의 순서가 달라집니다.**　맨 앞에서 본문을 하나만 읽어도 이제 14일이라는 내용을 확인할 수 있습니다.

```
1. library-1:0  도서 대출 기간은 14일입니다
2. library-1:1  .
```

**후보 집합과 재정렬 순위**　재정렬 전 첫 후보는 마침표 하나입니다. 대출 문장을 앞으로 보내도 전체 후보 수는 같습니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter38/index.html#practice-space)


### 셀 01 · 재정렬 전 후보 순위

후보 두 건을 그대로 받아 첫 번째 청크의 본문을 읽습니다.


In [1]:
candidates = [{'id': 'library-1:1', 'text': '.', 'source': '가상 도서관 이용 안내', 'page': 1},
 {'id': 'library-1:0',
  'text': '도서 대출 기간은 14일입니다',
  'source': '가상 도서관 이용 안내',
  'page': 1}]
print(candidates[0]['id'], repr(candidates[0]['text']))

library-1:1 '.'


마침표만 있는 library-1:1이 보입니다. 원문은 맞지만 이 조각만 받은 답변 함수에는 기간을 확인할 숫자가 없습니다.


### 셀 02 · sorted()의 key와 reverse

같은 후보 안에서 ‘대출’이 있는지 채점해 가장 앞에 놓습니다.


In [2]:
ranked = sorted(candidates, key=lambda doc: '대출' in doc['text'], reverse=True)
print(ranked[0]['id'], ranked[0]['text'])

library-1:0 도서 대출 기간은 14일입니다


library-1:0이 첫째가 되고 14일이 보입니다. 후보에 이 청크가 없었다면 재정렬만으로 새 문장을 가져올 수는 없습니다.


## 질의 확장과 청크 ID 중복 제거

‘책을 빌릴 수 있는 기간은?’을 세 낱말 벡터로 바꾸면 대출 항목이 0입니다. 이번에는 수업용 사전에 정한 표현 ‘대출 기간’을 더해 검색합니다. 원래 질문도 남겨 두고, 공백만 다른 표현은 한 번만 검색합니다.


### 과제 03 · 중복 질의 제거

원래 질문과 확장문을 순서대로 읽어 공백을 정리하고 중복 없이 남기세요.


In [3]:
queries = ['책을 빌릴 수 있는 기간은?', '대출 기간', '대출   기간']
# 첫 등장 순서를 지키며 중복을 제거하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>queries = [&#x27;책을 빌릴 수 있는 기간은?&#x27;, &#x27;대출 기간&#x27;, &#x27;대출   기간&#x27;]
unique = []
for query in queries:
    clean = &#x27; &#x27;.join(query.split())
    if clean not in unique:
        unique.append(clean)
print(unique)</code></pre>
<p>실행 결과</p><pre><code>[&#x27;책을 빌릴 수 있는 기간은?&#x27;, &#x27;대출 기간&#x27;]</code></pre>
<p>두 검색문이 남습니다. 검색문 중복을 줄였어도 양쪽 검색 결과에 같은 청크가 나올 수 있습니다. 결과를 합칠 때는 청크 ID 기준으로 한 번 더 확인합니다.</p>
</details>


### 셀 04 · 검색 결과의 청크 ID 합집합

같은 고정 인덱스에서 검색문별 결과를 받아 첫 등장 순서대로 합칩니다. 이 예의 검색은 ‘대출’ 포함 여부로 찾는 작은 로컬 검색입니다.

과제 03에서는 ‘책을 빌릴 수 있는 기간은?’을 출발점으로 표현을 늘렸습니다. 이번 셀은 두 표현 모두 ‘대출’을 포함하는 별도 예입니다. 여기서 관찰할 것은 새 문서 발견이 아니라 두 검색이 돌려준 같은 ID를 한 번만 남기는 과정입니다.


In [4]:
index = [{'id': 'library-1:0',
  'text': '도서 대출 기간은 14일입니다',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-1:1', 'text': '.', 'source': '가상 도서관 이용 안내', 'page': 1}]
search_queries = ['대출 기간은 며칠인가요?', '대출 기간']

def find_chunks(query):
    return [row['id'] for row in index if '대출' in query and '대출' in row['text']]

batches = [find_chunks(query) for query in search_queries]
merged = list(dict.fromkeys(chunk for batch in batches for chunk in batch))
print(batches)
print(merged)

[['library-1:0'], ['library-1:0']]
['library-1:0']


질문 표현 두 개가 같은 library-1:0을 찾았습니다. 검색은 두 번 했지만 답의 근거에는 청크 하나만 넘기면 됩니다. 원래 후보에 없던 청크가 추가되려면 확장한 검색문에서 실제로 다른 결과를 얻어야 합니다.


## 돌아보기

키워드 재순위가 성공한 위 사례는 의도적으로 순서를 뒤집은 연습입니다. 실제 채택 여부는 다음 장의 같은 평가 질문에서 나온 결과와 추가 작업 횟수로 판단하세요.

### 확장부터 최종 문서까지

**원질의와 확장 목록**　고정 규칙이 문구를 반환합니다. 실제 모델 생성이 아닙니다.

```
과제 03 · 책을 빌릴 수 있는 기간은?
확장: 대출 기간 / 대출   기간
```

**공백 정리와 중복 제거**　같은 문구는 한 번만 검색합니다. 총 질의 상한은 3개입니다.

```
책을 빌릴 수 있는 기간은?
대출 기간
```

**같은 인덱스에 각각 질의**　각 검색의 후보 상한은 2개입니다. 실패 시 부분 결과를 성공으로 넘기지 않습니다.

```
셀 04 · 대출 기간은 며칠인가요?
두 번째 검색어: 대출 기간
```

**청크 ID 중복 제거와 절단**　처음 나온 순서를 보존해 합친 뒤 최종 1개를 전달합니다.

```
[library-1:0] + [library-1:0]
중복 제거 → [library-1:0]
```
